# Qwen branch-aware tree rerank compare (multi-run Colab)

Этот ноутбук делает **несколько отдельных прогонов** branch-aware rerank,
но в **одной сессии и с одной загрузкой модели**.

Что он сравнивает:
- base scoring
- anchor rerank
- branch-aware tree rerank

Идея: не запускать один `CASE_FILTER`, а прогнать **сразу весь набор и отдельные family-runs** в одном notebook.


> Рекомендуемый runtime в Colab: **GPU**.
>
> Важный момент: модель грузится **один раз**, после чего notebook прогоняет все test suites по очереди.


In [ ]:
!nvidia-smi || true
!python --version
!pip install -q torch transformers einops pytest


In [ ]:
%cd /content
import os
if not os.path.exists('/content/ABPT'):
    !git clone https://github.com/kharkilirov1/Anchor-engine.git ABPT
%cd /content/ABPT
!git fetch --all
!git checkout main
!git pull --ff-only origin main


In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
DEVICE = 'cuda'
MAX_LENGTH = 256
FUTURE_WINDOW = 16
SPAN_THRESHOLD = 0.75
TOP_SPANS = 4
SEED = 7
RERANK_STRENGTH = 0.35
TREE_STRENGTH = 0.45

# Один notebook, одна загрузка модели, много отдельных прогонов.
RUN_SPECS = [
    {'name': 'all_cases', 'case_filter': ''},
    {'name': 'structural_core', 'case_filter': 'quantifier,proof_mode,api_framework'},
    {'name': 'quantifier', 'case_filter': 'quantifier'},
    {'name': 'proof_mode', 'case_filter': 'proof_mode'},
    {'name': 'api_framework', 'case_filter': 'api_framework'},
    {'name': 'induction', 'case_filter': 'induction'},
    {'name': 'instruction_constraints', 'case_filter': 'instruction_constraints'},
    {'name': 'entity_property', 'case_filter': 'entity_property'},
    {'name': 'legal_scope', 'case_filter': 'legal_scope'},
    {'name': 'units', 'case_filter': 'units'},
]

OUTPUT_DIR = 'archive/colab_tree_branch_runs'


In [ ]:
import json
from dataclasses import replace
from datetime import UTC, datetime
from pathlib import Path

import torch

from scripts.compare_qwen_tree_branch_rerank import (
    build_markdown_report,
    evaluate_case,
    summarize_results,
)
from src.data.qwen_rerank_cases import make_qwen_rerank_cases
from src.model.config import TOY_CONFIG
from src.model.qwen_anchor_overlay import QwenAnchorOverlay

torch.manual_seed(SEED)
cfg = replace(
    TOY_CONFIG,
    anchor_threshold=0.10,
    anchor_revision_threshold=0.35,
    anchor_contradiction_threshold=0.20,
    anchor_dead_end_threshold=0.50,
)

overlay = QwenAnchorOverlay.from_pretrained(
    model_name=MODEL_NAME,
    cfg=cfg,
    device=DEVICE,
    torch_dtype=torch.float16 if 'cuda' in DEVICE else None,
)
overlay.eval()

all_cases = make_qwen_rerank_cases()
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

print('loaded_model:', MODEL_NAME)
print('device:', DEVICE)
print('case_count_total:', len(all_cases))


In [ ]:
run_payloads = {}
run_reports = {}
run_summaries = []

for run_spec in RUN_SPECS:
    run_name = run_spec['name']
    case_filter = run_spec['case_filter']
    filters = {item.strip() for item in case_filter.split(',') if item.strip()}
    cases = [case for case in all_cases if not filters or case.family in filters or case.name in filters]

    print(f'\n===== RUN: {run_name} =====')
    print('filter:', case_filter or '<all>')
    print('cases:', len(cases))

    results = []
    for case in cases:
        result = evaluate_case(
            overlay=overlay,
            case=case,
            max_length=MAX_LENGTH,
            future_window=FUTURE_WINDOW,
            span_threshold=SPAN_THRESHOLD,
            top_spans=TOP_SPANS,
            rerank_strength=RERANK_STRENGTH,
            tree_strength=TREE_STRENGTH,
        )
        results.append(result)
        print(
            f"{case.name}: base={'ok' if result['base_correct'] else 'miss'} "
            f"anchor={'ok' if result['anchor_correct'] else 'miss'} "
            f"branch={'ok' if result['branch_correct'] else 'miss'} "
            f"branch_margin={result['branch_margin']:.4f}"
        )

    summary = summarize_results(results)
    payload = {
        'generated_at': datetime.now(UTC).isoformat(),
        'run_name': run_name,
        'case_filter': case_filter,
        'model': MODEL_NAME,
        'device': DEVICE,
        'max_length': MAX_LENGTH,
        'future_window': FUTURE_WINDOW,
        'span_threshold': SPAN_THRESHOLD,
        'top_spans': TOP_SPANS,
        'seed': SEED,
        'rerank_strength': RERANK_STRENGTH,
        'tree_strength': TREE_STRENGTH,
        'results': results,
        'summary': summary,
    }
    report = build_markdown_report(
        model_name=MODEL_NAME,
        device=DEVICE,
        max_length=MAX_LENGTH,
        future_window=FUTURE_WINDOW,
        rerank_strength=RERANK_STRENGTH,
        tree_strength=TREE_STRENGTH,
        results=results,
        summary=summary,
    )

    json_path = out_dir / f'{run_name}.json'
    md_path = out_dir / f'{run_name}.md'
    json_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    md_path.write_text(report, encoding='utf-8')

    run_payloads[run_name] = payload
    run_reports[run_name] = report
    run_summaries.append({
        'run_name': run_name,
        'case_filter': case_filter or '<all>',
        'case_count': summary['case_count'],
        'base_accuracy': summary['base_accuracy'],
        'anchor_accuracy': summary['anchor_accuracy'],
        'branch_accuracy': summary['branch_accuracy'],
        'branch_minus_base': summary['branch_minus_base_accuracy'],
        'branch_minus_anchor': summary['branch_minus_anchor_accuracy'],
        'rescued': ', '.join(summary['branch_rescued_cases']) if summary['branch_rescued_cases'] else 'none',
        'regressed': ', '.join(summary['branch_regressed_cases']) if summary['branch_regressed_cases'] else 'none',
        'json_path': str(json_path),
        'md_path': str(md_path),
    })


In [ ]:
for row in run_summaries:
    print('---')
    for key, value in row.items():
        print(f'{key}: {value}')


In [ ]:
for run_name in ['structural_core', 'quantifier', 'proof_mode', 'api_framework']:
    if run_name not in run_payloads:
        continue
    print(f'\n########## {run_name} ##########')
    payload = run_payloads[run_name]
    for item in payload['results']:
        print('===', item['name'], '===')
        print('base_correct:', item['base_correct'])
        print('anchor_correct:', item['anchor_correct'])
        print('branch_correct:', item['branch_correct'])
        print('base_margin:', item['base_margin'])
        print('anchor_margin:', item['anchor_margin'])
        print('branch_margin:', item['branch_margin'])
        print('preferred tree bonus:', item['preferred']['tree_bonus'])
        print('rejected tree bonus:', item['rejected']['tree_bonus'])
        print('preferred coverage:', item['preferred']['tree_coverage'])
        print('rejected coverage:', item['rejected']['tree_coverage'])
        print('preferred drift:', item['preferred']['tree_drift_score'])
        print('rejected drift:', item['rejected']['tree_drift_score'])
        print()


In [ ]:
RUN_TO_PREVIEW = 'structural_core'
print(run_reports[RUN_TO_PREVIEW][:12000])
